In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys 
sys.path.append('../src')

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [3]:
import torch
import tqdm
import matplotlib.pyplot as plt
from rescue.mapanything_pipeline import run_mapanything, save_points_to_glb, save_mesh_to_glb, save_language_features
from rescue.mapanything_pipeline import integrate_tsdf, save_colmap
from rescue.feature_reduction import TorchIncrementalPCA
from rescue.lang_features import LSegLangFeatures

device = "cuda" if torch.cuda.is_available() else "cpu"

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
predictions = run_mapanything('../generated/final.mp4', '../generated/final.json', fps = 2, use_ges=True)

Loading MapAnything model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading pretrained dinov2_vitg14 from torch hub


Using cache found in /home/srohatgi/.cache/torch/hub/facebookresearch_dinov2_main


Loading weights from local directory
Loading images (resolution_set=518, norm_type='dinov2')...
Running MapAnything model...
MapAnything model run complete...


In [5]:
# import open3d as o3d
# from rescue.mapanything_pipeline import integrate_tsdf, reproject_colors_onto_mesh


#     # predictions,
#     # voxel_length: float = 0.2,
#     # sdf_trunc: float = 0.8,
#     # depth_trunc: float = 30.0,
#     # block_count: int = 50000,
#     # conf_percentile: float = 10.0,
#     # outlier_nb_neighbors: int = 20,
#     # outlier_std_ratio: float = 2.0,
#     # smooth_iterations: int = 10,

# mesh = integrate_tsdf(predictions, voxel_length=0.05, 
# block_count=200000, smooth_iterations=2)
# mesh = reproject_colors_onto_mesh(mesh, predictions)


# o3d.io.write_triangle_mesh("../generated/reconstruction_tsdf.glb", mesh)

In [6]:
save_points_to_glb(predictions, "../generated/reconstruction_points.glb", show_cam = False)
save_mesh_to_glb(predictions, "../generated/reconstruction_mesh.glb", show_cam = False)

Building GLB scene
Using Pointmap Branch
GLB Scene built
Building GLB scene
Using Pointmap Branch
Creating mesh for multi-frame data...
GLB Scene built


In [7]:
# save_colmap(predictions, "../generated/colmap", export_points=True, export_images=True)
from rescue import utils 
import matplotlib.pyplot as plt
render_3d, depth = utils.render_3d_plot_from_above(recon_path="../generated/reconstruction_mesh.glb", bg_color=[0, 0, 0, 0])

Found num geometries: 101


In [8]:
lang_feats = LSegLangFeatures(checkpoint_path="../generated/lseg_minimal_e200.ckpt",)

In [9]:
img_list = [pred['img_no_norm'] for pred in predictions]
ipca = TorchIncrementalPCA(n_components=64, device='cuda')
img_feats = []

# First, extract all image features and store them before PCA
for img in tqdm.tqdm(img_list, desc="Extracting dense features"):
    img_perm = torch.permute(img, (0, 3, 1, 2))
    img_resized = torch.nn.functional.interpolate(
        img_perm, size=(480, 640), mode="bilinear", align_corners=False
    )
    img_feat = lang_feats.extract_dense_from_tensor(img_resized)
    img_feat = torch.nn.functional.interpolate(
        img_feat, size=(img.shape[1], img.shape[2]), mode="bilinear", align_corners=False
    )
    # (1, C, H, W) → (H, W, C)
    img_feat = torch.permute(img_feat.squeeze(0), (1, 2, 0))
    img_feats.append(img_feat)

# Fit the IPCA on each extracted feature map
for img_feat in tqdm.tqdm(img_feats, desc="Fitting IPCA"):
    # img_feat: (H, W, C) → flatten to (H*W, C)
    ipca.partial_fit(img_feat.view(-1, img_feat.shape[-1]))

# Transform each feature map with the fitted IPCA and reshape
language_features = []
for img_feat in tqdm.tqdm(img_feats, desc="Transforming with IPCA"):
    shape_hw = img_feat.shape[:2]
    flat_feat = img_feat.view(-1, img_feat.shape[-1])
    transformed = ipca.transform(flat_feat)  # (H*W, n_components)
    transformed = transformed.view(*shape_hw, -1)  # (H, W, n_components)
    language_features.append(transformed)

language_features = torch.stack(language_features, dim=0)  # (S, H, W, C)

Extracting dense features:   0%|          | 0/101 [00:00<?, ?it/s]

Transforming with IPCA: 100%|██████████| 101/101 [00:00<00:00, 3691.65it/s]


In [10]:
save_language_features(predictions, language_features, "../generated/language_features.safetensors", ipca)

[MapAnything] Saved 15364600 points to ../generated/language_features.safetensors
